# Single Bias-Experiment Runner

Runs one bias-evaluation experiment end-to-end. Accepts either a YAML config file (`--config`) or a full set of CLI arguments specifying the bias type (length / position / sycophancy / uncertainty), reward model, dataset source, and output directories. Loads the corresponding `BiasExperiment` subclass, trains a linear probe on a held-out probe set, evaluates on the test split, saves the probe artifact, and writes diagnostic plots. Also handles cross-dataset generalisation by pointing the probe trainer at one dataset and the evaluator at another.


In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path
from typing import Type

import yaml

PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.nb.experiments.base import BiasExperiment, ExperimentConfig, ExperimentResults
from src.nb.experiments.length import LengthBiasExperiment
from src.nb.experiments.sycophancy import SycophancyBiasExperiment
from src.nb.experiments.uncertainty import UncertaintyBiasExperiment
from src.nb.experiments.position import (
    PositionBiasExperiment, 
    BinaryPositionBiasExperiment,
    FreeformPositionBiasExperiment,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

EXPERIMENT_CLASSES: dict[str, Type[BiasExperiment]] = {
    "length": LengthBiasExperiment,
    "sycophancy": SycophancyBiasExperiment,
    "uncertainty": UncertaintyBiasExperiment,
    "position": PositionBiasExperiment,
}


In [ ]:
def _load_defaults(section: str) -> dict:
    """Return default config values for *section* from configs/default_values.yaml."""
    with open(PROJECT_ROOT / "configs" / "default_values.yaml") as f:
        import yaml as _yaml
        return _yaml.safe_load(f).get(section, {})


In [ ]:
def run(config_path: "str | Path | None" = None, **overrides):
    """Run a bias evaluation experiment.

    Args:
        config_path: Path to an experiment YAML config.  If relative, resolved
                     from the project-root ``configs/`` directory.
        **overrides: Per-run keyword overrides, e.g. ``device="cpu"``.

    Returns:
        ExperimentResults
    """
    cfg = _load_defaults("run_experiment")

    if config_path is not None:
        p = Path(config_path)
        if not p.is_absolute():
            p = PROJECT_ROOT / "configs" / p
        with open(p) as f:
            cfg.update(yaml.safe_load(f))

    cfg.update(overrides)

    config = ExperimentConfig.from_dict(cfg)

    logger.info("Experiment : %s", config.name)
    logger.info("Bias type  : %s", config.bias_type)
    logger.info("Model      : %s", config.model_path)
    logger.info("Dataset    : %s", config.dataset_source)

    exp_cls = EXPERIMENT_CLASSES.get(config.bias_type)
    if exp_cls is None:
        raise ValueError(
            f"Unknown bias type: {config.bias_type!r}. "
            f"Choose from: {list(EXPERIMENT_CLASSES)}"
        )

    if config.bias_type == "position":
        if config.dataset_class in (
            "position_freeform",
            "position_freeform_bigbench",
            "position_freeform_plausibleqa",
        ):
            exp_cls = FreeformPositionBiasExperiment
            logger.info("Using freeform position bias experiment")
        elif config.dataset_class == "position_plausibleqa" or (
            "plausibleqa" in (config.dataset_source or "").lower()
            and not config.dataset_class
        ):
            exp_cls = BinaryPositionBiasExperiment
            logger.info("Using binary position bias experiment for PlausibleQA")

    if cfg.get("probe_source"):
        logger.info(
            "Cross-dataset: probe from %s, eval on %s",
            cfg["probe_source"], config.dataset_source,
        )
        config.extra["probe_source"] = cfg["probe_source"]
        if cfg.get("probe_extra"):
            config.extra["probe_extra"] = cfg["probe_extra"]

    experiment = exp_cls(config)
    return experiment.run()


In [ ]:
run("configs/length_skywork.yaml")
